In [28]:
import pandas as pd
import numpy as np
from datasets import Dataset

In [29]:
train_df = pd.read_csv('/kaggle/input/competitions/tst-day-3-upsolving/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/tst-day-3-upsolving/test.csv')


In [30]:
train_df['code'].apply(lambda x: len(x))

0       1248
1       1213
2        619
3        960
4       1186
        ... 
1882     574
1883    1007
1884     925
1885     773
1886     745
Name: code, Length: 1887, dtype: int64

In [31]:
import pandas as pd
import re

keywords = [
    "dp", "dynamic programming", "dfs", "bfs", 
    "dijkstra", "recursion", "binary search", "greedy", 'binary', 'pointers' 'depth-first search', 'Depth First Search',
    'tree', 'subtree', 'subtrees', 'breadth first search', 'breadth-first search', 'hash',
    'bitwise','XOR','map', 'deque', 'linked list', 'mod', 'heap', 'unordered_map', 'priority queue',
    'binary tree', 'sliding window', 'union-find', 'union', 'bit', 'bits', 'unions', 'multiset',
    'edge', 'node', 'hashmap', 'random',
]

def add_keyword_features(df, text_column, word_list):
    # Копируем таблицу, чтобы не менять оригинал случайно
    df_result = df.copy()
    
    for word in word_list:
        # Создаем паттерн: \b означает границу слова
        # re.escape нужен, если в слове есть спецсимволы (например, C++)
        pattern = rf'\b{re.escape(word)}\b'
        
        # Создаем новую колонку (называем её по слову)
        # Ищем слово в тексте (без учета регистра) и превращаем в 0/1
        column_name = f"feat_{word.replace(' ', '_')}"
        df_result[column_name] = df_result[text_column].str.contains(
            pattern, case=False, na=False, regex=True
        ).astype(int)
        
    return df_result

# Применение

In [32]:
from sklearn.preprocessing import *
encoder = LabelEncoder()

data = train_df.rename(columns ={'difficulty': 'label', 'code': 'text'} )
data['label'] = encoder.fit_transform(data['label'] )

data = data.drop('id',axis=1)
# data.index=train_df.id
dataset = Dataset.from_dict(data)
dataset
data

key_words = add_keyword_features(data, 'text', keywords)
key_words = key_words.drop(['label','text'], axis=1)


In [33]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification, TrainingArguments, Trainer

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
def tokenize_func(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_func, batched=True)

Map:   0%|          | 0/1887 [00:00<?, ? examples/s]

In [34]:
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=3)

# Настройки обучения
training_args = TrainingArguments(
    output_dir="./results",          # Где сохранять модель
    # num_train_epochs=3,              # Сколько раз пройти по данным
    per_device_train_batch_size=8,   # Размер батча
    logging_steps=10,
    eval_strategy="no",        # Можно добавить валидационный сет
    # learning_rate=2e-5,              # Стандартный шаг для BERT
    fp16=True,
    num_train_epochs=5,
    learning_rate=3e-5,
    weight_decay=0.01
)

# Инициализируем Trainer
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_datasets,
# )

# # ЗАПУСК ОБУЧЕНИЯ
# trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [35]:
from sklearn.utils.class_weight import compute_class_weight
import torch.nn as nn
import torch
from transformers import Trainer  # оставляем тот же Trainer как базовый

# 1) Вычисляем веса классов на основе train меток (замена/добавление)
# используем исходный DataFrame `data`, в котором уже есть колонка 'label'
labels = data['label'].values
class_weights_np = compute_class_weight(class_weight="balanced",
                                        classes=np.unique(labels),
                                        y=labels)
# конвертируем в тензор — будем перемещать на устройство при вычислении loss
class_weights = torch.tensor(class_weights_np, dtype=torch.float)

# --- КОММЕНТАРИЙ: class_weights теперь содержит вес для каждого класса,
# например [w_easy, w_medium, w_hard]. Эти веса будут использоваться в loss. ---

class WeightedTrainer(Trainer):
    # 👇 добавили **kwargs (ВАЖНО для новой версии transformers)
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):

        labels = inputs.get("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )

        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss
# 3) Создаём экземпляр нашего WeightedTrainer (замена предыдущего trainer = Trainer(...))
trainer = WeightedTrainer(   #можно было бы просто Trainer оставить для обычной тренки
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    # если нужно, можно добавить eval_dataset=... и data_collator=...
)

# Запускаем обучение как раньше
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
10,1.105277
20,1.101157
30,1.113794
40,1.099163
50,1.105660
60,1.095994
70,1.097081
80,1.107809
90,1.097151
100,1.086124


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=590, training_loss=0.930565915673466, metrics={'train_runtime': 181.2198, 'train_samples_per_second': 52.064, 'train_steps_per_second': 3.256, 'total_flos': 620618774065920.0, 'train_loss': 0.930565915673466, 'epoch': 5.0})

In [36]:
test_df
data_test = test_df.rename(columns ={'code': 'text'} )
data_test = data_test.drop('id',axis=1)
# data.index=train_df.id
dataset_test = Dataset.from_dict(data_test)
dataset_test

Dataset({
    features: ['text'],
    num_rows: 472
})

In [37]:
import torch
import torch.nn.functional as F
from tqdm import tqdm # для визуализации прогресса
import numpy as np

# 1. Очистим память от предыдущих попыток
torch.cuda.empty_cache()

ttext = list(data['text'])
batch_size = 16  # Уменьши до 8 или 4, если ошибка повторится
all_probabilities = []

model.eval()

ans_frombert = []
with torch.no_grad():
    # Проходим по текстам мелкими шагами
    for i in tqdm(range(0, len(ttext), batch_size)):
        batch_texts = ttext[i : i + batch_size]
        
        # Токенизируем текущий батч
        inputs = tokenizer(
            batch_texts, 
            return_tensors="pt", 
            padding=True, 
            truncation=True, 
            max_length=512
        ).to(model.device)
        
        # Получаем логиты
        logits = model(**inputs).logits
        
        # Считаем вероятности (шансы)
        probs = F.softmax(logits, dim=1).cpu().numpy()
        all_probabilities.append(probs)
        
        # ВЫБИРАЕМ ЛУЧШИЙ КЛАСС (вместо softmax)
        # argmax находит индекс максимального числа в каждой строке
        # pred = torch.argmax(logits, dim=1).cpu().numpy()
        # ans_frombert.extend(pred)

# Собираем всё в один финальный массив
# final_probs = np.vstack(all_probabilities)

100%|██████████| 118/118 [00:32<00:00,  3.65it/s]


In [38]:

new_train = pd.DataFrame(np.vstack(all_probabilities))
new_train

,0,1,2
0,0.230118,0.166532,0.603350
1,0.170904,0.042380,0.786716
2,0.889025,0.018491,0.092484
3,0.110438,0.308524,0.581038
4,0.175807,0.229220,0.594974
...,...,...,...
1882,0.062071,0.336867,0.601062
1883,0.082610,0.157261,0.760129
1884,0.067883,0.243753,0.688364
1885,0.040793,0.445648,0.513558


In [39]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words='english')
# pd.DataFrame(tfidf.fit_transform(data_test['text']).toarray())
# tfidf.fit(data['text'])
vec = pd.DataFrame(tfidf.fit_transform(data['text']).toarray(), columns = [f'col_{i}' for i in range(10000)] )
# vec

In [40]:
new_train = pd.concat([key_words, new_train,vec], axis=1)
# new_train = pd.concat([key_words, new_train], axis=1)   #без тфдф

# new_train = vec #потом убрать
new_train['len'] = data['text'].apply(lambda x: len(x))
new_train['newline_count'] = data['text'].str.count('\n')
new_train

,feat_dp,feat_dynamic_programming,feat_dfs,feat_bfs,feat_dijkstra,feat_recursion,feat_binary_search,feat_greedy,feat_binary,feat_pointersdepth-first_search,...,col_9992,col_9993,col_9994,col_9995,col_9996,col_9997,col_9998,col_9999,len,newline_count
0,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1248,4
1,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1213,14
2,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,619,2
3,0,0,0,0,0,0,0,0,1,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,960,2
4,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1186,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1882,1,1,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,574,2
1883,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1007,4
1884,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,925,5
1885,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,773,0


In [41]:
from xgboost import XGBClassifier
model_xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    objective='multi:softprob', # для мультиклассовой классификации
    num_class=3,
    tree_method='hist',         # Оптимизация для больших данных
    device="cuda"               # Используем GPU, если доступно
)
model_xgb.fit(new_train,data['label'])

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device='cuda', early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=None, num_class=3, ...)

test prediction

In [42]:
data_test

,text
0,The algorithm iterates from 2 to n using a loo...
1,We use Depth-First Search for traversing the b...
2,The algorithm used is a dynamic programming ap...
3,1. Create a map (or dictionary) to store the c...
4,The algorithm is a straightforward brute force...
...,...
467,The algorithm iteratively checks if it's possi...
468,The algorithm makes use of the given programmi...
469,The algorithm for this problem is dynamic prog...
470,The algorithm relies on a helper function (`le...


In [43]:
import torch
import torch.nn.functional as F
from tqdm import tqdm # для визуализации прогресса
import numpy as np

# 1. Очистим память от предыдущих попыток
torch.cuda.empty_cache()

ttext = list(data_test['text'])
batch_size = 16  # Уменьши до 8 или 4, если ошибка повторится
all_probabilities2 = []

model.eval()

ans_frombert = []
with torch.no_grad():
    # Проходим по текстам мелкими шагами
    for i in tqdm(range(0, len(ttext), batch_size)):
        batch_texts = ttext[i : i + batch_size]
        
        # Токенизируем текущий батч
        inputs = tokenizer(
            batch_texts, 
            return_tensors="pt", 
            padding=True, 
            truncation=True, 
            max_length=512
        ).to(model.device)
        
        # Получаем логиты
        logits = model(**inputs).logits
        
        # Считаем вероятности (шансы)
        probs = F.softmax(logits, dim=1).cpu().numpy()
        all_probabilities2.append(probs)
        
        # ВЫБИРАЕМ ЛУЧШИЙ КЛАСС (вместо softmax)
        # argmax находит индекс максимального числа в каждой строке
        # pred = torch.argmax(logits, dim=1).cpu().numpy()
        # ans_frombert.extend(pred)

# Собираем всё в один финальный массив
final_probs2 = np.vstack(all_probabilities2)

100%|██████████| 30/30 [00:07<00:00,  3.80it/s]


In [44]:

new_test = pd.DataFrame(np.vstack(all_probabilities2))
new_test

,0,1,2
0,0.060232,0.646311,0.293458
1,0.323175,0.074593,0.602232
2,0.050612,0.362338,0.587050
3,0.795729,0.034716,0.169555
4,0.018150,0.745379,0.236471
...,...,...,...
467,0.882103,0.019898,0.097999
468,0.096583,0.580565,0.322853
469,0.023032,0.814929,0.162039
470,0.170978,0.203395,0.625627


In [45]:

# pd.DataFrame(tfidf.fit_transform(data_test['text']).toarray())
vec2 = pd.DataFrame(tfidf.transform(data_test['text']).toarray(), columns = [f'col_{i}' for i in range(10000)] )
# vec2

In [46]:

key_words_test = add_keyword_features(data_test, 'text', keywords)
key_words_test = key_words_test.drop(['text'], axis=1)

new_test = pd.concat([key_words_test, new_test,vec2], axis=1)
# new_test = pd.concat([key_words_test, new_test], axis=1) #без тфдф


# new_test = vec2 #потом убрать
new_test['len'] = data_test['text'].apply(lambda x: len(x))
new_test['newline_count'] = data_test['text'].str.count('\n')

new_test

,feat_dp,feat_dynamic_programming,feat_dfs,feat_bfs,feat_dijkstra,feat_recursion,feat_binary_search,feat_greedy,feat_binary,feat_pointersdepth-first_search,...,col_9992,col_9993,col_9994,col_9995,col_9996,col_9997,col_9998,col_9999,len,newline_count
0,0,0,0,0,0,1,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,466,0
1,0,0,1,0,0,0,0,0,1,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,506,0
2,1,1,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1004,2
3,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,694,5
4,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,540,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
467,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,663,4
468,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,942,6
469,1,1,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1099,7
470,0,0,0,0,0,1,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1043,3


In [47]:
new_train.shape

(1887, 10043)

In [48]:
y_pred = model_xgb.predict(new_test)

In [49]:
subm = pd.read_csv('/kaggle/input/competitions/tst-day-3-upsolving/sample_submission.csv')
subm['difficulty'] = encoder.inverse_transform(y_pred)
subm

,id,difficulty
0,1013,medium
1,1218,medium
2,1716,medium
3,1464,easy
4,2227,hard
...,...,...
467,1889,easy
468,2545,medium
469,2370,hard
470,386,medium


In [50]:
test_df

,id,code
0,1013,The algorithm iterates from 2 to n using a loo...
1,1218,We use Depth-First Search for traversing the b...
2,1716,The algorithm used is a dynamic programming ap...
3,1464,1. Create a map (or dictionary) to store the c...
4,2227,The algorithm is a straightforward brute force...
...,...,...
467,1889,The algorithm iteratively checks if it's possi...
468,2545,The algorithm makes use of the given programmi...
469,2370,The algorithm for this problem is dynamic prog...
470,386,The algorithm relies on a helper function (`le...


In [51]:

subm.to_csv('answer.csv', index=False)
print('a3')

a3


In [52]:
(pd.DataFrame(np.vstack(all_probabilities2))[2]<.6).sum()

np.int64(381)

In [53]:
# for i in list(train_df[(train_df['difficulty']=='hard') | (train_df['difficulty']=='hard')]['code'][:100]):
#     print(i)
#     print()
#     print()
#     print()
    